In [1]:
%pip install -q requests pandas "numpy<2.0" scikit-learn matplotlib gymnasium shimmy tqdm rich "stable-baselines3[extra]"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import ast
import json
import os
import random
import time
from typing import Any, Dict, List, Optional, Tuple

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 200)

GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE = "https://clob.polymarket.com"

CONFIG = {
    "keyword": None,
    "max_markets": 500,
    "page_limit": 500,
    "active": True,
    "closed": False,
    "min_liquidity": 1000.0,
    "min_volume": 1000.0,
    "interval": "max",
    "fidelity": 60,
    "start_ts": None,
    "end_ts": None,
    "outdir": "polymarket_output_rl_tuned",
    "sleep_sec": 0.1,
}

RL_CONFIG = {
    "lookback": 32,
    "allow_short": False,
    "transaction_cost": 0.0025,
    "initial_cash": 1000.0,
    "action_bins": 5,
    "min_trade_fraction": 0.10,
    "fixed_trade_penalty": 0.0002,
    "risk_aversion": 0.0002,
    "clip_reward": 0.05,
    "timesteps": 5_000_000,
    "learning_rate": 1e-4,
    "n_steps": 4096,
    "batch_size": 512,
    "gamma": 0.995,
    "gae_lambda": 0.98,
    "ent_coef": 0.005,
    "vf_coef": 0.7,
    "clip_range": 0.2,
    "verbose": 1,
    "seed": 42,
    "device": "cuda",
    "train_frac": 0.70,
    "valid_frac": 0.15,
    "num_envs": 8,
    "checkpoint_freq": 100_000,
    "eval_freq": 50_000,
    "policy_hidden_sizes": [256, 256, 128],
}

os.makedirs(CONFIG["outdir"], exist_ok=True)
os.makedirs(os.path.join(CONFIG["outdir"], "per_market"), exist_ok=True)
os.makedirs(os.path.join(CONFIG["outdir"], "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(CONFIG["outdir"], "best_model"), exist_ok=True)

CONFIG, RL_CONFIG

({'keyword': None,
  'max_markets': 500,
  'page_limit': 500,
  'active': True,
  'closed': False,
  'min_liquidity': 1000.0,
  'min_volume': 1000.0,
  'interval': 'max',
  'fidelity': 60,
  'start_ts': None,
  'end_ts': None,
  'outdir': 'polymarket_output_rl_tuned',
  'sleep_sec': 0.1},
 {'lookback': 32,
  'allow_short': False,
  'transaction_cost': 0.0025,
  'initial_cash': 1000.0,
  'action_bins': 5,
  'min_trade_fraction': 0.1,
  'fixed_trade_penalty': 0.0002,
  'risk_aversion': 0.0002,
  'clip_reward': 0.05,
  'timesteps': 5000000,
  'learning_rate': 0.0001,
  'n_steps': 4096,
  'batch_size': 512,
  'gamma': 0.995,
  'gae_lambda': 0.98,
  'ent_coef': 0.005,
  'vf_coef': 0.7,
  'clip_range': 0.2,
  'verbose': 1,
  'seed': 42,
  'device': 'cuda',
  'train_frac': 0.7,
  'valid_frac': 0.15,
  'num_envs': 8,
  'checkpoint_freq': 100000,
  'eval_freq': 50000,
  'policy_hidden_sizes': [256, 256, 128]})

In [3]:
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

if not hasattr(np, "bool"):
    np.bool = bool

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback, EvalCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv

In [4]:
def make_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,
        read=5,
        connect=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.headers.update(
        {
            "User-Agent": "polymarket-dataset-pipeline-notebook/1.0",
            "Accept": "application/json",
        }
    )
    return session

SESSION = make_session()

def parse_maybe_json_array(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            parsed = json.loads(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
    return []

def safe_float(x: Any, default: float = np.nan) -> float:
    try:
        if x is None or x == "":
            return default
        return float(x)
    except Exception:
        return default

def safe_str(x: Any) -> str:
    return "" if x is None else str(x)

def get_json(url: str, params: Optional[Dict[str, Any]] = None) -> Any:
    r = SESSION.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [5]:
def fetch_markets_page(
    limit: int = 100,
    offset: int = 0,
    active: Optional[bool] = True,
    closed: Optional[bool] = False,
    order: str = "volume",
    ascending: bool = False,
) -> List[Dict[str, Any]]:
    params: Dict[str, Any] = {
        "limit": limit,
        "offset": offset,
        "order": order,
        "ascending": str(ascending).lower(),
    }
    if active is not None:
        params["active"] = str(active).lower()
    if closed is not None:
        params["closed"] = str(closed).lower()

    return get_json(f"{GAMMA_BASE}/markets", params=params)

def discover_markets(
    keyword: Optional[str],
    max_markets: int,
    page_limit: int = 100,
    active: Optional[bool] = True,
    closed: Optional[bool] = False,
    min_liquidity: float = 0.0,
    min_volume: float = 0.0,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    offset = 0
    keyword_lower = keyword.lower() if keyword else None

    while len(rows) < max_markets:
        page = fetch_markets_page(
            limit=page_limit,
            offset=offset,
            active=active,
            closed=closed,
            order="volume",
            ascending=False,
        )
        if not page:
            break

        for m in page:
            question = safe_str(m.get("question", ""))
            description = safe_str(m.get("description", ""))
            category = safe_str(m.get("category", ""))
            subcategory = safe_str(m.get("subcategory", ""))
            text_blob = " ".join([question, description, category, subcategory]).lower()

            if keyword_lower and keyword_lower not in text_blob:
                continue

            liquidity = safe_float(m.get("liquidity"), 0.0)
            volume = safe_float(m.get("volume"), 0.0)
            if liquidity < min_liquidity or volume < min_volume:
                continue

            clob_token_ids = parse_maybe_json_array(m.get("clobTokenIds"))
            outcomes = parse_maybe_json_array(m.get("outcomes"))
            outcome_prices = parse_maybe_json_array(m.get("outcomePrices"))
            
            if len(clob_token_ids) != 2:
                continue
            
            if len(outcomes) != 2:
                continue

            if not clob_token_ids:
                continue

            yes_token_id = safe_str(clob_token_ids[0]) if len(clob_token_ids) >= 1 else ""
            no_token_id = safe_str(clob_token_ids[1]) if len(clob_token_ids) >= 2 else ""

            row = {
                "market_id": safe_str(m.get("id")),
                "question": question,
                "slug": safe_str(m.get("slug")),
                "category": category,
                "subcategory": subcategory,
                "description": description,
                "liquidity": liquidity,
                "volume": volume,
                "active": m.get("active"),
                "closed": m.get("closed"),
                "archived": m.get("archived"),
                "startDate": safe_str(m.get("startDate")),
                "endDate": safe_str(m.get("endDate")),
                "yes_token_id": yes_token_id,
                "no_token_id": no_token_id,
                "outcomes": json.dumps(outcomes, ensure_ascii=False),
                "outcomePrices": json.dumps(outcome_prices, ensure_ascii=False),
            }
            rows.append(row)

            if len(rows) >= max_markets:
                break

        offset += page_limit

    return pd.DataFrame(rows).drop_duplicates(subset=["market_id"]).reset_index(drop=True)

In [6]:
def fetch_price_history(
    token_id: str,
    interval: str = "1h",
    fidelity: int = 60,
    start_ts: Optional[int] = None,
    end_ts: Optional[int] = None,
) -> pd.DataFrame:
    params: Dict[str, Any] = {
        "market": token_id,
        "interval": interval,
        "fidelity": fidelity,
    }
    if start_ts is not None:
        params["startTs"] = start_ts
    if end_ts is not None:
        params["endTs"] = end_ts

    data = get_json(f"{CLOB_BASE}/prices-history", params=params)

    history = data.get("history", [])
    if not history:
        return pd.DataFrame(columns=["timestamp", "price"])

    df = pd.DataFrame(history)
    if "t" not in df.columns or "p" not in df.columns:
        return pd.DataFrame(columns=["timestamp", "price"])

    df = df.rename(columns={"t": "timestamp", "p": "price"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["timestamp", "price"]).sort_values("timestamp").reset_index(drop=True)
    return df

def download_dataset(
    markets_df: pd.DataFrame,
    outdir: str,
    interval: str,
    fidelity: int,
    start_ts: Optional[int],
    end_ts: Optional[int],
    sleep_sec: float = 0.1,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    metadata_path = os.path.join(outdir, "markets.csv")
    prices_path = os.path.join(outdir, "prices.csv")

    all_prices: List[pd.DataFrame] = []
    kept_meta: List[Dict[str, Any]] = []

    for i, row in markets_df.iterrows():
        token_id = row["yes_token_id"]
        if not token_id:
            continue

        try:
            hist = fetch_price_history(
                token_id=token_id,
                interval=interval,
                fidelity=fidelity,
                start_ts=start_ts,
                end_ts=end_ts,
            )
        except Exception as e:
            print(f"[WARN] failed history for market_id={row['market_id']} token={token_id}: {e}")
            continue

        print(f"market_id={row['market_id']} token={token_id} history_rows={len(hist)}")
        if hist.empty or len(hist) < 30:
            print(f"[INFO] skipping market_id={row['market_id']} because history is too short")
            continue

        hist["market_id"] = row["market_id"]
        hist["yes_token_id"] = token_id
        hist["question"] = row["question"]
        hist["category"] = row["category"]
        hist["subcategory"] = row["subcategory"]
        hist["liquidity"] = row["liquidity"]
        hist["volume"] = row["volume"]
        hist["endDate"] = row["endDate"]

        per_market_path = os.path.join(outdir, "per_market", f"{row['market_id']}.csv")
        hist.to_csv(per_market_path, index=False)

        all_prices.append(hist)
        kept_meta.append(row.to_dict())

        print(
            f"[OK] {i + 1}/{len(markets_df)} "
            f"market_id={row['market_id']} rows={len(hist)} "
            f"question={row['question'][:80]}"
        )
        time.sleep(sleep_sec)

    metadata_df = pd.DataFrame(kept_meta).reset_index(drop=True)
    prices_df = pd.concat(all_prices, ignore_index=True) if all_prices else pd.DataFrame()

    metadata_df.to_csv(metadata_path, index=False)
    prices_df.to_csv(prices_path, index=False)

    return metadata_df, prices_df

In [7]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ts = pd.to_datetime(df["timestamp"], utc=True)
    df["hour"] = ts.dt.hour
    df["dayofweek"] = ts.dt.dayofweek
    df["dayofmonth"] = ts.dt.day
    return df

def build_features(prices_df: pd.DataFrame) -> pd.DataFrame:
    if prices_df.empty:
        return pd.DataFrame()

    df = prices_df.copy()
    df = df.sort_values(["market_id", "timestamp"]).reset_index(drop=True)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    def per_market(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values("timestamp").copy()

        g["ret_1"] = g["price"].pct_change(1)
        g["ret_3"] = g["price"].pct_change(3)
        g["ret_6"] = g["price"].pct_change(6)
        g["ret_12"] = g["price"].pct_change(12)

        g["mom_3"] = g["price"] - g["price"].shift(3)
        g["mom_6"] = g["price"] - g["price"].shift(6)
        g["mom_12"] = g["price"] - g["price"].shift(12)

        g["rolling_mean_6"] = g["price"].rolling(6).mean()
        g["rolling_mean_12"] = g["price"].rolling(12).mean()
        g["rolling_std_6"] = g["price"].rolling(6).std()
        g["rolling_std_12"] = g["price"].rolling(12).std()

        g["zscore_6"] = (g["price"] - g["rolling_mean_6"]) / g["rolling_std_6"]
        g["zscore_12"] = (g["price"] - g["rolling_mean_12"]) / g["rolling_std_12"]

        p = g["price"].clip(1e-5, 1 - 1e-5)
        g["logit_price"] = np.log(p / (1 - p))

        g["future_price"] = g["price"].shift(-1)
        g["future_ret_1"] = g["future_price"] / g["price"] - 1.0
        g["target_up"] = (g["future_ret_1"] > 0).astype(int)

        return g

    feat = df.groupby("market_id", group_keys=False).apply(per_market)
    feat = add_time_features(feat)
    feat = feat.dropna(subset=["future_ret_1", "target_up"]).copy()
    feat = feat.reset_index(drop=True)

    valid_numeric = feat[
        [
            "ret_1",
            "ret_3",
            "ret_6",
            "mom_3",
            "mom_6",
            "rolling_mean_6",
            "rolling_std_6",
            "logit_price",
        ]
    ].notna().sum(axis=1) > 0
    feat = feat.loc[valid_numeric].copy()

    ordered_cols = [
        "timestamp", "question", "yes_token_id", "target_up", "future_ret_1",
        "price", "ret_1", "ret_3", "ret_6", "ret_12",
        "mom_3", "mom_6", "mom_12",
        "rolling_mean_6", "rolling_mean_12",
        "rolling_std_6", "rolling_std_12",
        "zscore_6", "zscore_12", "logit_price",
        "liquidity", "volume", "hour", "dayofweek", "dayofmonth",
        "category", "subcategory", "market_id"
    ]
    ordered_cols = [c for c in ordered_cols if c in feat.columns]
    return feat[ordered_cols].copy()

def time_split(
    feat_df: pd.DataFrame,
    train_frac: float = 0.70,
    valid_frac: float = 0.15,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = feat_df.sort_values("timestamp").reset_index(drop=True)
    n = len(df)
    train_end = int(n * train_frac)
    valid_end = int(n * (train_frac + valid_frac))
    train_df = df.iloc[:train_end].copy()
    valid_df = df.iloc[train_end:valid_end].copy()
    test_df = df.iloc[valid_end:].copy()
    return train_df, valid_df, test_df

In [8]:
def fit_baseline_model(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    outdir: str,
) -> Dict[str, Any]:
    target_col = "target_up"

    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    train_df["question"] = train_df["question"].fillna("")
    valid_df["question"] = valid_df["question"].fillna("")
    test_df["question"] = test_df["question"].fillna("")

    numeric_features = [
        "price",
        "ret_1",
        "ret_3",
        "ret_6",
        "ret_12",
        "mom_3",
        "mom_6",
        "mom_12",
        "rolling_mean_6",
        "rolling_mean_12",
        "rolling_std_6",
        "rolling_std_12",
        "zscore_6",
        "zscore_12",
        "logit_price",
        "liquidity",
        "volume",
        "hour",
        "dayofweek",
        "dayofmonth",
    ]

    categorical_features = ["category", "subcategory", "market_id"]
    text_feature = "question"
    feature_cols = numeric_features + categorical_features + ['question']

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(int).copy()

    X_valid = valid_df[feature_cols].copy()
    y_valid = valid_df[target_col].astype(int).copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(int).copy()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
            (
                "text",
                TfidfVectorizer(
                    max_features=1000,
                    ngram_range=(1, 2),
                    stop_words="english",
                ),
                "question",
            ),
        ]
    )

    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42,
    )

    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    pipe.fit(X_train, y_train)

    valid_proba = pipe.predict_proba(X_valid)[:, 1]
    valid_pred = (valid_proba >= 0.5).astype(int)

    test_proba = pipe.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= 0.5).astype(int)

    metrics = {
        "valid_accuracy": float(accuracy_score(y_valid, valid_pred)) if len(y_valid) else np.nan,
        "test_accuracy": float(accuracy_score(y_test, test_pred)) if len(y_test) else np.nan,
        "valid_auc": float(roc_auc_score(y_valid, valid_proba)) if len(np.unique(y_valid)) > 1 else np.nan,
        "test_auc": float(roc_auc_score(y_test, test_proba)) if len(np.unique(y_test)) > 1 else np.nan,
        "n_train": int(len(train_df)),
        "n_valid": int(len(valid_df)),
        "n_test": int(len(test_df)),
        "markets_train": int(train_df["market_id"].nunique()),
        "markets_valid": int(valid_df["market_id"].nunique()),
        "markets_test": int(test_df["market_id"].nunique()),
    }

    with open(os.path.join(outdir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    with open(os.path.join(outdir, "valid_classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(classification_report(y_valid, valid_pred, digits=4))

    with open(os.path.join(outdir, "test_classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(classification_report(y_test, test_pred, digits=4))

    valid_out = valid_df[["timestamp", "question", "market_id", "target_up", "future_ret_1"]].copy()
    valid_out["pred_proba_up"] = valid_proba
    valid_out["pred_up"] = valid_pred
    valid_out.to_csv(os.path.join(outdir, "valid_predictions.csv"), index=False)

    test_out = test_df[["timestamp", "question", "market_id", "target_up", "future_ret_1"]].copy()
    test_out["pred_proba_up"] = test_proba
    test_out["pred_up"] = test_pred
    test_out.to_csv(os.path.join(outdir, "test_predictions.csv"), index=False)

    return metrics

In [9]:
markets_df = discover_markets(
    keyword=CONFIG["keyword"],
    max_markets=CONFIG["max_markets"],
    page_limit=CONFIG["page_limit"],
    active=CONFIG["active"],
    closed=CONFIG["closed"],
    min_liquidity=CONFIG["min_liquidity"],
    min_volume=CONFIG["min_volume"],
)

markets_df.to_csv(os.path.join(CONFIG["outdir"], "discovered_markets.csv"), index=False)
print(f"Discovered {len(markets_df)} markets")
markets_df.head(10)

Discovered 499 markets


,market_id,question,slug,category,subcategory,description,liquidity,volume,active,closed,archived,startDate,endDate,yes_token_id,no_token_id,outcomes,outcomePrices
0,701726,"Will Hyperliquid reach $100 by December 31, 2026?",will-hyperliquid-reach-100-by-december-31-2026,,,This market will immediately resolve to “Yes” ...,13182.16240,99920.010542,True,False,False,2025-11-24T21:09:54.093719Z,2027-01-01T05:00:00Z,5563253116425987458153208646309676537567512231...,8200389029322785814444369062112915175131090753...,"[""Yes"", ""No""]","[""0.215"", ""0.785""]"
1,1572665,Will Sophia Chikirou advance to the second rou...,will-sophia-chikirou-advance-to-the-second-rou...,,,Municipal elections are scheduled to be held i...,7655.27119,99870.852249,True,False,False,2026-03-13T20:38:32.252617Z,2026-03-15T00:00:00Z,8706457581334293780618009225366654363232945897...,1105570629929405601757662858737525131274474307...,"[""Yes"", ""No""]","[""0.9895"", ""0.0105""]"
2,1602050,VCU Rams vs. North Carolina Tar Heels,cbb-vcu-ncar-2026-03-19,,,"In the upcoming CBB game, scheduled for March ...",81784.00820,9976.279046,True,False,False,2026-03-15T23:03:15.48647Z,2026-03-19T04:00:00Z,8400736124175897848933019720282723570028907889...,6272267179000548293620665438226214625237349562...,"[""VCU Rams"", ""North Carolina Tar Heels""]","[""0.425"", ""0.575""]"
3,1591769,Over $12M committed to the P2P Protocol public...,over-12m-committed-to-the-p2p-protocol-public-...,,,This market will resolve to “Yes” if total com...,2512.26440,9975.973757,True,False,False,2026-03-14T21:34:21.893Z,2026-07-01T04:00:00Z,9225300277532587048629139682480493265214633474...,1885919778921466838271165583781722270087746420...,"[""Yes"", ""No""]","[""0.37"", ""0.63""]"
4,637014,Will Khaled Mashal win the Nobel Peace Prize i...,will-khaled-mashal-win-the-nobel-peace-prize-i...,,,This market will resolve according to the winn...,32336.44221,99751.336413,True,False,False,2025-10-16T22:30:37.055538Z,2026-10-10T00:00:00Z,4817558919700430121831697879056774909752184660...,8718060268156357528336886756762911747500274167...,"[""Yes"", ""No""]","[""0.0045"", ""0.9955""]"
5,1508633,Will FedEx (FDX) beat quarterly earnings?,fdx-quarterly-earnings-nongaap-eps-03-19-2026-...,,,"As of market creation, FedEx is estimated to r...",3518.98580,9974.678856,True,False,False,2026-03-05T17:29:22.655823Z,2026-03-19T21:00:00Z,1042391260369502866632394239175928959586345495...,1083239144752523541872367250761062451264229748...,"[""Yes"", ""No""]","[""0.935"", ""0.065""]"
6,1138909,Masoud Pezeshkian out by December 31?,masoud-pezeshkian-out-by-december-31,,,"This market will resolve to ""Yes"" if the Presi...",11562.17200,99733.677593,True,False,False,2026-01-08T23:55:48.750544Z,2026-12-31T00:00:00Z,2147884814668554283617019600216497883083738090...,6940916877416575789714840524748560996575602373...,"[""Yes"", ""No""]","[""0.425"", ""0.575""]"
7,1469764,Will Ahmad Hosseini Khorasani be head of state...,will-ahmad-hosseini-khorasani-be-head-of-state...,,,This market will resolve to the individual who...,21775.74607,9970.557949,True,False,False,2026-03-01T00:18:26.28Z,2026-12-31T00:00:00Z,3048248432246166029147846179711140980961385763...,1306016185809624321958931969454195157085558985...,"[""Yes"", ""No""]","[""0.009"", ""0.991""]"
8,1623299,Will Elon Musk post 540-559 tweets from March ...,elon-musk-of-tweets-march-20-march-27-540-559,,,This market will resolve according to the numb...,26323.15857,9967.285485,True,False,False,2026-03-17T04:10:38.916062Z,2026-03-27T16:00:00Z,3067743349795700042726308147610119581123650632...,8520270951799751479389134226982506852511134368...,"[""Yes"", ""No""]","[""0.0075"", ""0.9925""]"
9,676791,Deel IPO before 2027?,deel-ipo-before-2027,,,"This market will resolve to ""Yes"" if the liste...",4220.84120,99661.822013,True,False,False,2025-11-12T21:28:43.84235Z,2026-12-31T00:00:00Z,1032528607001344912728239738123994613079723041...,2428720360848145572270826767845808222911115658...,"[""Yes"", "

In [10]:
metadata_df, prices_df = download_dataset(
    markets_df=markets_df,
    outdir=CONFIG["outdir"],
    interval=CONFIG["interval"],
    fidelity=CONFIG["fidelity"],
    start_ts=CONFIG["start_ts"],
    end_ts=CONFIG["end_ts"],
    sleep_sec=CONFIG["sleep_sec"],
)

print(f"Kept {len(metadata_df)} markets")
print(f"Downloaded {len(prices_df)} price rows")
prices_df.head()

market_id=701726 token=55632531164259874581532086463096765375675122310464288883906870061905029632548 history_rows=672
[OK] 1/499 market_id=701726 rows=672 question=Will Hyperliquid reach $100 by December 31, 2026?
market_id=1572665 token=87064575813342937806180092253666543632329458979432586376676088759901734231755 history_rows=78
[OK] 2/499 market_id=1572665 rows=78 question=Will Sophia Chikirou advance to the second round of the 2026 Paris municipal ele
market_id=1602050 token=84007361241758978489330197202827235700289078898880514910472901778979242019662 history_rows=42
[OK] 3/499 market_id=1602050 rows=42 question=VCU Rams vs. North Carolina Tar Heels
market_id=1591769 token=92253002775325870486291396824804932652146334743548025985493394591638559052006 history_rows=68
[OK] 4/499 market_id=1591769 rows=68 question=Over $12M committed to the P2P Protocol public sale?
market_id=637014 token=4817558919700430121831697879056774909752184660255737582917693916506064447935 history_rows=652
[OK] 

,timestamp,price,market_id,yes_token_id,question,category,subcategory,liquidity,volume,endDate
0,2026-02-17 18:00:35+00:00,0.180,701726,5563253116425987458153208646309676537567512231...,"Will Hyperliquid reach $100 by December 31, 2026?",,,13182.1624,99920.010542,2027-01-01T05:00:00Z
1,2026-02-17 19:00:32+00:00,0.180,701726,5563253116425987458153208646309676537567512231...,"Will Hyperliquid reach $100 by December 31, 2026?",,,13182.1624,99920.010542,2027-01-01T05:00:00Z
2,2026-02-17 20:00:17+00:00,0.185,701726,5563253116425987458153208646309676537567512231...,"Will Hyperliquid reach $100 by December 31, 2026?",,,13182.1624,99920.010542,2027-01-01T05:00:00Z
3,2026-02-17 21:00:23+00:00,0.185,701726,5563253116425987458153208646309676537567512231...,"Will Hyperliquid reach $100 by December 31, 2026?",,,13182.1624,99920.010542,2027-01-01T05:00:00Z
4,2026-02-17 22:00:19+00:00,0.185,701726,5563253116425987458153208646309676537567512231...,"Will Hyperliquid reach $100 by December 31, 2026?",,,13182.1624,99920.010542,2027-01-01T05:00:00Z


In [11]:
print("Unique markets:", prices_df["market_id"].nunique())
print("Min timestamp:", prices_df["timestamp"].min())
print("Max timestamp:", prices_df["timestamp"].max())
print("Rows per market summary:")
prices_df.groupby("market_id").size().describe()

Unique markets: 472
Min timestamp: 2026-02-17 18:00:26+00:00
Max timestamp: 2026-03-17 17:59:35+00:00
Rows per market summary:


count    472.000000
mean     516.631356
std      217.057145
min       33.000000
25%      363.000000
50%      641.000000
75%      669.000000
max      672.000000
dtype: float64

In [12]:
metadata_df[["market_id", "question", "category", "liquidity", "volume", "endDate"]].head(20)

,market_id,question,category,liquidity,volume,endDate
0,701726,"Will Hyperliquid reach $100 by December 31, 2026?",,13182.16240,99920.010542,2027-01-01T05:00:00Z
1,1572665,Will Sophia Chikirou advance to the second rou...,,7655.27119,99870.852249,2026-03-15T00:00:00Z
2,1602050,VCU Rams vs. North Carolina Tar Heels,,81784.00820,9976.279046,2026-03-19T04:00:00Z
3,1591769,Over $12M committed to the P2P Protocol public...,,2512.26440,9975.973757,2026-07-01T04:00:00Z
4,637014,Will Khaled Mashal win the Nobel Peace Prize i...,,32336.44221,99751.336413,2026-10-10T00:00:00Z
5,1508633,Will FedEx (FDX) beat quarterly earnings?,,3518.98580,9974.678856,2026-03-19T21:00:00Z
6,1138909,Masoud Pezeshkian out by December 31?,,11562.17200,99733.677593,2026-12-31T00:00:00Z
7,1469764,Will Ahmad Hosseini Khorasani be head of state...,,21775.74607,9970.557949,2026-12-31T00:00:00Z
8,676791,Deel IPO before 2027?,,4220.84120,99661.822013,2026-12-31T00:00:00Z
9,679268,Will Juliana Stratton be the Democratic nomine...,,13007.79760,99570.468947,2026-03-17T00:00:00Z


In [13]:
feat_df = build_features(prices_df)
feat_df.to_csv(os.path.join(CONFIG["outdir"], "features.csv"), index=False)

print("Feature rows:", len(feat_df))
feat_df.head()

Feature rows: 243378


,timestamp,question,yes_token_id,target_up,future_ret_1,price,ret_1,ret_3,ret_6,ret_12,mom_3,mom_6,mom_12,rolling_mean_6,rolling_mean_12,rolling_std_6,rolling_std_12,zscore_6,zscore_12,logit_price,liquidity,volume,hour,dayofweek,dayofmonth,category,subcategory,market_id
0,2026-02-17 18:00:32+00:00,Will the Bharatiya Janata Party (BJP) win the ...,1011427917241379403799484981250977225770917971...,0,0.0,0.0035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.651486,5959.80386,8610.168405,18,1,17,,,1003791
1,2026-02-17 19:00:31+00:00,Will the Bharatiya Janata Party (BJP) win the ...,1011427917241379403799484981250977225770917971...,0,0.0,0.0035,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.651486,5959.80386,8610.168405,19,1,17,,,1003791
2,2026-02-17 20:00:16+00:00,Will the Bharatiya Janata Party (BJP) win the ...,1011427917241379403799484981250977225770917971...,0,0.0,0.0035,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.651486,5959.80386,8610.168405,20,1,17,,,1003791
3,2026-02-17 21:00:21+00:00,Will the Bharatiya Janata Party (BJP) win the ...,1011427917241379403799484981250977225770917971...,0,0.0,0.0035,0.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.651486,5959.80386,8610.168405,21,1,17,,,1003791
4,2026-02-17 22:00:17+00:00,Will the Bharatiya Janata Party (BJP) win the ...,1011427917241379403799484981250977225770917971...,0,0.0,0.0035,0.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.651486,5959.80386,8610.168405,22,1,17,,,1003791


In [14]:
def build_rl_features(prices_df: pd.DataFrame, lookback: int = 20) -> pd.DataFrame:
    df = prices_df.copy()
    df = df.sort_values(["market_id", "timestamp"]).reset_index(drop=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["liquidity"] = pd.to_numeric(df["liquidity"], errors="coerce").fillna(0.0)
    df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0.0)
    df["endDate_dt"] = pd.to_datetime(df["endDate"], errors="coerce", utc=True)

    def per_market(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values("timestamp").copy()
        g["ret_1"] = g["price"].pct_change(1).fillna(0.0)
        g["ret_2"] = g["price"].pct_change(2).fillna(0.0)
        g["ret_3"] = g["price"].pct_change(3).fillna(0.0)
        g["ret_6"] = g["price"].pct_change(6).fillna(0.0)
        g["ret_12"] = g["price"].pct_change(12).fillna(0.0)

        g["mom_3"] = (g["price"] - g["price"].shift(3)).fillna(0.0)
        g["mom_6"] = (g["price"] - g["price"].shift(6)).fillna(0.0)

        g["rolling_mean_6"] = g["price"].rolling(6).mean()
        g["rolling_mean_12"] = g["price"].rolling(12).mean()
        g["rolling_std_6"] = g["price"].rolling(6).std()
        g["rolling_std_12"] = g["price"].rolling(12).std()

        g["rolling_mean_6"] = g["rolling_mean_6"].fillna(method="bfill").fillna(g["price"])
        g["rolling_mean_12"] = g["rolling_mean_12"].fillna(method="bfill").fillna(g["price"])
        g["rolling_std_6"] = g["rolling_std_6"].fillna(method="bfill").fillna(0.0)
        g["rolling_std_12"] = g["rolling_std_12"].fillna(method="bfill").fillna(0.0)

        g["zscore_6"] = ((g["price"] - g["rolling_mean_6"]) / (g["rolling_std_6"] + 1e-8)).fillna(0.0)
        g["zscore_12"] = ((g["price"] - g["rolling_mean_12"]) / (g["rolling_std_12"] + 1e-8)).fillna(0.0)

        p = g["price"].clip(1e-5, 1 - 1e-5)
        g["logit_price"] = np.log(p / (1 - p))

        if g["endDate_dt"].notna().any():
            g["time_to_expiry_hours"] = ((g["endDate_dt"] - g["timestamp"]).dt.total_seconds() / 3600.0).clip(lower=0.0)
        else:
            g["time_to_expiry_hours"] = 0.0

        g["next_price"] = g["price"].shift(-1)
        g["next_ret"] = (g["next_price"] / g["price"] - 1.0).replace([np.inf, -np.inf], 0.0).fillna(0.0)

        g["market_age_steps"] = np.arange(len(g))
        g["market_length"] = len(g)

        return g

    feat = df.groupby("market_id", group_keys=False).apply(per_market).reset_index(drop=True)

    keep_cols = [
        "timestamp", "market_id", "question", "category", "subcategory",
        "price", "next_price", "next_ret", "ret_1", "ret_2", "ret_3", "ret_6", "ret_12",
        "mom_3", "mom_6", "rolling_mean_6", "rolling_mean_12",
        "rolling_std_6", "rolling_std_12", "zscore_6", "zscore_12",
        "logit_price", "liquidity", "volume", "time_to_expiry_hours",
        "market_age_steps", "market_length"
    ]
    feat = feat[keep_cols].copy()

    numeric_cols = [
        "price", "next_price", "next_ret", "ret_1", "ret_2", "ret_3", "ret_6", "ret_12",
        "mom_3", "mom_6", "rolling_mean_6", "rolling_mean_12",
        "rolling_std_6", "rolling_std_12", "zscore_6", "zscore_12",
        "logit_price", "liquidity", "volume", "time_to_expiry_hours",
        "market_age_steps", "market_length"
    ]
    feat[numeric_cols] = feat[numeric_cols].replace([np.inf, -np.inf], 0.0).fillna(0.0)

    feat["question"] = feat["question"].fillna("").astype(str)
    feat["category"] = feat["category"].fillna("unknown").astype(str)
    feat["subcategory"] = feat["subcategory"].fillna("unknown").astype(str)

    counts = feat.groupby("market_id").size()
    valid_ids = counts[counts >= max(lookback + 5, 30)].index
    feat = feat[feat["market_id"].isin(valid_ids)].copy()

    return feat.reset_index(drop=True)

def split_market_history(
    market_df: pd.DataFrame,
    train_frac: float = 0.70,
    valid_frac: float = 0.15,
    lookback: int = 20,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    g = market_df.sort_values("timestamp").reset_index(drop=True)
    n = len(g)
    train_end = int(n * train_frac)
    valid_end = int(n * (train_frac + valid_frac))

    train_df = g.iloc[:train_end].copy()
    valid_df = g.iloc[max(0, train_end - lookback):valid_end].copy()
    test_df = g.iloc[max(0, valid_end - lookback):].copy()
    return train_df, valid_df, test_df


def build_market_splits(
    rl_df: pd.DataFrame,
    train_frac: float,
    valid_frac: float,
    lookback: int,
) -> Tuple[Dict[str, pd.DataFrame], Dict[str, pd.DataFrame], Dict[str, pd.DataFrame]]:
    train_markets: Dict[str, pd.DataFrame] = {}
    valid_markets: Dict[str, pd.DataFrame] = {}
    test_markets: Dict[str, pd.DataFrame] = {}

    for market_id, g in rl_df.groupby("market_id"):
        train_df, valid_df, test_df = split_market_history(
            g, train_frac=train_frac, valid_frac=valid_frac, lookback=lookback
        )
        if len(train_df) >= lookback + 5:
            train_markets[market_id] = train_df.reset_index(drop=True)
        if len(valid_df) >= lookback + 5:
            valid_markets[market_id] = valid_df.reset_index(drop=True)
        if len(test_df) >= lookback + 5:
            test_markets[market_id] = test_df.reset_index(drop=True)

    return train_markets, valid_markets, test_markets


rl_df = build_rl_features(prices_df, lookback=RL_CONFIG["lookback"])
train_markets, valid_markets, test_markets = build_market_splits(
    rl_df,
    train_frac=RL_CONFIG["train_frac"],
    valid_frac=RL_CONFIG["valid_frac"],
    lookback=RL_CONFIG["lookback"],
)

print("RL rows:", len(rl_df))
print("Train markets:", len(train_markets))
print("Valid markets:", len(valid_markets))
print("Test markets:", len(test_markets))

rl_df.head()

RL rows: 243817
Train markets: 459
Valid markets: 461
Test markets: 471


,timestamp,market_id,question,category,subcategory,price,next_price,next_ret,ret_1,ret_2,ret_3,ret_6,ret_12,mom_3,mom_6,rolling_mean_6,rolling_mean_12,rolling_std_6,rolling_std_12,zscore_6,zscore_12,logit_price,liquidity,volume,time_to_expiry_hours,market_age_steps,market_length
0,2026-02-17 18:00:32+00:00,1003791,Will the Bharatiya Janata Party (BJP) win the ...,,,0.0035,0.0035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0035,0.0035,0.0,0.0,0.0,0.0,-5.651486,5959.80386,8610.168405,1949.991111,0,670
1,2026-02-17 19:00:31+00:00,1003791,Will the Bharatiya Janata Party (BJP) win the ...,,,0.0035,0.0035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0035,0.0035,0.0,0.0,0.0,0.0,-5.651486,5959.80386,8610.168405,1948.991389,1,670
2,2026-02-17 20:00:16+00:00,1003791,Will the Bharatiya Janata Party (BJP) win the ...,,,0.0035,0.0035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0035,0.0035,0.0,0.0,0.0,0.0,-5.651486,5959.80386,8610.168405,1947.995556,2,670
3,2026-02-17 21:00:21+00:00,1003791,Will the Bharatiya Janata Party (BJP) win the ...,,,0.0035,0.0035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0035,0.0035,0.0,0.0,0.0,0.0,-5.651486,5959.80386,8610.168405,1946.994167,3,670
4,2026-02-17 22:00:17+00:00,1003791,Will the Bharatiya Janata Party (BJP) win the ...,,,0.0035,0.0035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0035,0.0035,0.0,0.0,0.0,0.0,-5.651486,5959.80386,8610.168405,1945.995278,4,670


In [15]:
class PolymarketTradingEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        markets: Dict[str, pd.DataFrame],
        lookback: int = 20,
        transaction_cost: float = 0.002,
        risk_aversion: float = 0.0005,
        clip_reward: float = 0.2,
        allow_short: bool = False,
        initial_cash: float = 1000.0,
        action_bins: int = 11,
        min_trade_fraction: float = 0.0,
        fixed_trade_penalty: float = 0.0,
        deterministic_market_order: bool = False,
        seed: int = 42,
    ):
        super().__init__()

        self.markets = {
            str(k): v.sort_values("timestamp").reset_index(drop=True).copy()
            for k, v in markets.items()
            if len(v) >= lookback + 5
        }

        self.market_ids = list(self.markets.keys())
        self.lookback = lookback
        self.transaction_cost = transaction_cost
        self.risk_aversion = risk_aversion
        self.clip_reward = clip_reward
        self.allow_short = allow_short
        self.initial_cash = float(initial_cash)
        self.action_bins = int(action_bins)
        self.min_trade_fraction = float(min_trade_fraction)
        self.fixed_trade_penalty = float(fixed_trade_penalty)
        self.deterministic_market_order = deterministic_market_order
        self.seed_value = seed

        self.rng = np.random.default_rng(seed)
        self.market_cursor = 0

        self.extra_feature_names = [
            "price",
            "no_price",
            "logit_price",
            "ret_1",
            "ret_3",
            "ret_6",
            "rolling_std_6",
            "zscore_6",
            "time_to_expiry_hours",
            "liquidity",
            "volume",
            "cash_frac",
            "yes_frac",
            "no_frac",
            "cash_to_initial",
            "yes_qty",
            "no_qty",
            "nav_to_initial",
        ]
        obs_dim = self.lookback + len(self.extra_feature_names)

        self.action_space = spaces.MultiDiscrete([5, self.action_bins])
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(obs_dim,),
            dtype=np.float32,
        )

        self.current_market_id: Optional[str] = None
        self.current_df: Optional[pd.DataFrame] = None
        self.current_step = self.lookback
        self.cash = self.initial_cash
        self.yes_qty = 0.0
        self.no_qty = 0.0
        self.equity = self.initial_cash
        self.done = False

    def _select_market_id(self) -> str:
        if self.deterministic_market_order:
            market_id = self.market_ids[self.market_cursor % len(self.market_ids)]
            self.market_cursor += 1
            return market_id
        return str(self.rng.choice(self.market_ids))

    def _current_prices(self) -> Tuple[float, float]:
        assert self.current_df is not None
        row = self.current_df.iloc[self.current_step]
        yes_price = float(np.clip(row["price"], 1e-6, 1.0 - 1e-6))
        no_price = float(np.clip(1.0 - yes_price, 1e-6, 1.0 - 1e-6))
        return yes_price, no_price

    def _portfolio_value(self, yes_price: float, no_price: float) -> float:
        return float(self.cash + self.yes_qty * yes_price + self.no_qty * no_price)

    def _get_obs(self) -> np.ndarray:
        assert self.current_df is not None
        start = self.current_step - self.lookback
        end = self.current_step
        window = self.current_df.iloc[start:end]

        returns_window = window["ret_1"].to_numpy(dtype=np.float32)
        if len(returns_window) < self.lookback:
            pad = np.zeros(self.lookback - len(returns_window), dtype=np.float32)
            returns_window = np.concatenate([pad, returns_window])

        row = self.current_df.iloc[self.current_step]
        yes_price, no_price = self._current_prices()
        nav = self._portfolio_value(yes_price, no_price)
        nav_denom = max(nav, 1e-8)

        cash_frac = self.cash / nav_denom
        yes_frac = (self.yes_qty * yes_price) / nav_denom
        no_frac = (self.no_qty * no_price) / nav_denom

        extras = np.array([
            float(yes_price),
            float(no_price),
            float(row["logit_price"]),
            float(row["ret_1"]),
            float(row["ret_3"]),
            float(row["ret_6"]),
            float(row["rolling_std_6"]),
            float(row["zscore_6"]),
            float(row["time_to_expiry_hours"]) / 24.0,
            np.log1p(float(row["liquidity"])),
            np.log1p(float(row["volume"])),
            float(cash_frac),
            float(yes_frac),
            float(no_frac),
            float(self.cash / max(self.initial_cash, 1e-8)),
            float(self.yes_qty),
            float(self.no_qty),
            float(nav / max(self.initial_cash, 1e-8)),
        ], dtype=np.float32)

        obs = np.concatenate([returns_window, extras]).astype(np.float32)
        obs = np.nan_to_num(obs, nan=0.0, posinf=0.0, neginf=0.0)
        return obs

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.current_market_id = self._select_market_id()
        self.current_df = self.markets[self.current_market_id]
        self.current_step = self.lookback
        self.cash = self.initial_cash
        self.yes_qty = 0.0
        self.no_qty = 0.0
        yes_price, no_price = self._current_prices()
        self.equity = self._portfolio_value(yes_price, no_price)
        self.done = False

        obs = self._get_obs()
        info = {
            "market_id": self.current_market_id,
            "timestamp": self.current_df.iloc[self.current_step]["timestamp"],
        }
        return obs, info

    def step(self, action):
        assert self.current_df is not None

        action_arr = np.asarray(action).reshape(-1)
        action_type = int(action_arr[0])
        size_bin = int(action_arr[1]) if len(action_arr) > 1 else 0
        max_bin = max(self.action_bins - 1, 1)
        size_fraction = float(np.clip(size_bin / max_bin, 0.0, 1.0))

        if size_fraction < self.min_trade_fraction:
            action_type = 0
            size_fraction = 0.0

        yes_price, no_price = self._current_prices()
        nav_before = self._portfolio_value(yes_price, no_price)

        trade_notional = 0.0
        fees_paid = 0.0
        executed_qty = 0.0
        forced_liquidation_notional = 0.0
        forced_liquidation_fees = 0.0

        if action_type == 1:
            if self.no_qty > 0.0:
                gross_proceeds = self.no_qty * no_price
                liq_fee = gross_proceeds * self.transaction_cost
                self.cash += max(gross_proceeds - liq_fee, 0.0)
                forced_liquidation_notional += gross_proceeds
                forced_liquidation_fees += liq_fee
                self.no_qty = 0.0

            spend = size_fraction * self.cash
            if spend > 0.0:
                fees_paid = spend * self.transaction_cost
                effective_spend = max(spend - fees_paid, 0.0)
                executed_qty = effective_spend / max(yes_price, 1e-8)
                self.cash -= spend
                self.yes_qty += executed_qty
                trade_notional = spend

        elif action_type == 2:
            if self.yes_qty > 0.0:
                gross_proceeds = self.yes_qty * yes_price
                liq_fee = gross_proceeds * self.transaction_cost
                self.cash += max(gross_proceeds - liq_fee, 0.0)
                forced_liquidation_notional += gross_proceeds
                forced_liquidation_fees += liq_fee
                self.yes_qty = 0.0

            spend = size_fraction * self.cash
            if spend > 0.0:
                fees_paid = spend * self.transaction_cost
                effective_spend = max(spend - fees_paid, 0.0)
                executed_qty = effective_spend / max(no_price, 1e-8)
                self.cash -= spend
                self.no_qty += executed_qty
                trade_notional = spend

        elif action_type == 3:
            qty = size_fraction * self.yes_qty
            if qty > 0.0:
                gross_proceeds = qty * yes_price
                fees_paid = gross_proceeds * self.transaction_cost
                self.cash += max(gross_proceeds - fees_paid, 0.0)
                self.yes_qty -= qty
                executed_qty = qty
                trade_notional = gross_proceeds

        elif action_type == 4:
            qty = size_fraction * self.no_qty
            if qty > 0.0:
                gross_proceeds = qty * no_price
                fees_paid = gross_proceeds * self.transaction_cost
                self.cash += max(gross_proceeds - fees_paid, 0.0)
                self.no_qty -= qty
                executed_qty = qty
                trade_notional = gross_proceeds

        fees_paid += forced_liquidation_fees
        trade_notional += forced_liquidation_notional

        self.current_step += 1
        if self.current_step >= len(self.current_df) - 1:
            self.done = True

        yes_price_next, no_price_next = self._current_prices()
        nav_after = self._portfolio_value(yes_price_next, no_price_next)
        pnl = nav_after - nav_before
        reward = pnl / max(nav_before, 1e-8)

        if self.risk_aversion > 0.0:
            exposure_value = self.yes_qty * yes_price_next + self.no_qty * no_price_next
            leverage_like = exposure_value / max(nav_after, 1e-8)
            risk_penalty = self.risk_aversion * leverage_like
            reward -= risk_penalty
        else:
            risk_penalty = 0.0

        trade_penalty = self.fixed_trade_penalty if trade_notional > 0.0 else 0.0
        reward -= trade_penalty

        if self.clip_reward is not None:
            reward = float(np.clip(reward, -self.clip_reward, self.clip_reward))

        self.equity = nav_after

        obs = self._get_obs()
        info = {
            "market_id": self.current_market_id,
            "timestamp": self.current_df.iloc[self.current_step]["timestamp"],
            "price": yes_price_next,
            "no_price": no_price_next,
            "next_ret": float(self.current_df.iloc[self.current_step]["next_ret"]),
            "category": str(self.current_df.iloc[self.current_step].get("category", "")),
            "action_type": action_type,
            "size_bin": size_bin,
            "size_fraction": size_fraction,
            "cash": float(self.cash),
            "yes_qty": float(self.yes_qty),
            "no_qty": float(self.no_qty),
            "trade_notional": float(trade_notional),
            "fees_paid": float(fees_paid),
            "executed_qty": float(executed_qty),
            "forced_liquidation_notional": float(forced_liquidation_notional),
            "portfolio_value": float(nav_after),
            "reward": float(reward),
            "risk_penalty": float(risk_penalty),
            "trade_penalty": float(trade_penalty),
        }
        return obs, float(reward), self.done, False, info

In [16]:

def make_train_env(rank: int = 0):
    def _init() -> PolymarketTradingEnv:
        return PolymarketTradingEnv(
            markets=train_markets,
            lookback=RL_CONFIG["lookback"],
            transaction_cost=RL_CONFIG["transaction_cost"],
            risk_aversion=RL_CONFIG["risk_aversion"],
            clip_reward=RL_CONFIG["clip_reward"],
            allow_short=RL_CONFIG["allow_short"],
            deterministic_market_order=False,
            initial_cash=RL_CONFIG["initial_cash"],
            action_bins=RL_CONFIG["action_bins"],
            min_trade_fraction=RL_CONFIG["min_trade_fraction"],
            fixed_trade_penalty=RL_CONFIG["fixed_trade_penalty"],
            seed=RL_CONFIG["seed"] + rank,
        )
    return _init


def make_eval_env(markets_dict: Dict[str, pd.DataFrame]) -> PolymarketTradingEnv:
    return PolymarketTradingEnv(
        markets=markets_dict,
        lookback=RL_CONFIG["lookback"],
        transaction_cost=RL_CONFIG["transaction_cost"],
        risk_aversion=RL_CONFIG["risk_aversion"],
        clip_reward=RL_CONFIG["clip_reward"],
        allow_short=RL_CONFIG["allow_short"],
        deterministic_market_order=True,
        initial_cash=RL_CONFIG["initial_cash"],
        action_bins=RL_CONFIG["action_bins"],
        min_trade_fraction=RL_CONFIG["min_trade_fraction"],
        fixed_trade_penalty=RL_CONFIG["fixed_trade_penalty"],
        seed=RL_CONFIG["seed"],
    )


vec_train_env = DummyVecEnv([make_train_env(rank=i) for i in range(RL_CONFIG["num_envs"])])
vec_valid_env = DummyVecEnv([lambda: make_eval_env(valid_markets)])
print(f"Training env ready with {RL_CONFIG['num_envs']} parallel episodes.")

Training env ready with 8 parallel episodes.


In [17]:
policy_kwargs = dict(
    net_arch=dict(
        pi=RL_CONFIG["policy_hidden_sizes"],
        vf=RL_CONFIG["policy_hidden_sizes"],
    )
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(RL_CONFIG["checkpoint_freq"] // RL_CONFIG["num_envs"], 1),
    save_path=os.path.join(CONFIG["outdir"], "checkpoints"),
    name_prefix="ppo_polymarket",
)

eval_callback = EvalCallback(
    vec_valid_env,
    best_model_save_path=os.path.join(CONFIG["outdir"], "best_model"),
    log_path=os.path.join(CONFIG["outdir"], "eval_logs"),
    eval_freq=max(RL_CONFIG["eval_freq"] // RL_CONFIG["num_envs"], 1),
    deterministic=True,
    render=False,
)

callback = CallbackList([checkpoint_callback, eval_callback])

ppo_model = PPO(
    policy="MlpPolicy",
    env=vec_train_env,
    learning_rate=RL_CONFIG["learning_rate"],
    n_steps=RL_CONFIG["n_steps"],
    batch_size=RL_CONFIG["batch_size"],
    gamma=RL_CONFIG["gamma"],
    gae_lambda=RL_CONFIG["gae_lambda"],
    ent_coef=RL_CONFIG["ent_coef"],
    vf_coef=RL_CONFIG["vf_coef"],
    clip_range=RL_CONFIG["clip_range"],
    verbose=RL_CONFIG["verbose"],
    seed=RL_CONFIG["seed"],
    tensorboard_log=os.path.join(CONFIG["outdir"], "tensorboard"),
    device=RL_CONFIG["device"],
    policy_kwargs=policy_kwargs,
)

ppo_model.learn(
    total_timesteps=RL_CONFIG["timesteps"],
    progress_bar=True,
    callback=callback,
)

final_model_path = os.path.join(CONFIG["outdir"], "ppo_polymarket_trader")
ppo_model.save(final_model_path)
print(f"Saved final PPO model to {final_model_path}")

best_model_path = os.path.join(CONFIG["outdir"], "best_model", "best_model.zip")
if os.path.exists(best_model_path):
    best_model = PPO.load(best_model_path, device=RL_CONFIG["device"])
    print(f"Loaded best validation model from {best_model_path}")
else:
    best_model = ppo_model

Using cuda device
Logging to polymarket_output_rl_tuned/tensorboard/PPO_7


Output()

/home/kevinshi/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(
/home/kevinshi/jupyterlab/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


------------------------------
| time/              |       |
|    fps             | 3731  |
|    iterations      | 1     |
|    time_elapsed    | 8     |
|    total_timesteps | 32768 |
------------------------------


/home/kevinshi/jupyterlab/.venv/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:70: 
UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting 
modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first
with ``Monitor`` wrapper.
  warnings.warn(

...

------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 55.2         |
|    mean_reward          | 0            |
| time/                   |              |
|    total_timesteps      | 4950000      |
| train/                  |              |
|    approx_kl            | 0.0049452297 |
|    clip_fraction        | 0.0256       |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.64        |
|    explained_variance   | 0.278        |
|    learning_rate        | 0.0001       |
|    loss                 | -0.0206      |
|    n_updates            | 1510         |
|    policy_gradient_loss | -0.00744     |
|    value_loss           | 3.93e-05     |
------------------------------------------
--------------------------------
| time/              |         |
|    fps             | 2715    |
|    iterations      | 152     |
|    time_elapsed    | 1833    |
|    total_timesteps | 4980736 |
----------------------------

Eval num_timesteps=5000000, episode_reward=0.00 +/- 0.00

Episode length: 55.60 +/- 1.62

------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 55.6         |
|    mean_reward          | 0            |
| time/                   |              |
|    total_timesteps      | 5000000      |
| train/                  |              |
|    approx_kl            | 0.0057260096 |
|    clip_fraction        | 0.0233       |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.63        |
|    explained_variance   | -0.222       |
|    learning_rate        | 0.0001       |
|    loss                 | 0.036        |
|    n_updates            | 1520         |
|    policy_gradient_loss | -0.00315     |
|    value_loss           | 1.99e-05     |
------------------------------------------
--------------------------------
| time/              |         |
|    fps             | 2711    |
|    iterations      | 153     |
|    time_elapsed    | 1849    |
|    total_timesteps | 5013504 |
----------------------------

Saved final PPO model to polymarket_output_rl_tuned/ppo_polymarket_trader
Loaded best validation model from polymarket_output_rl_tuned/best_model/best_model.zip


In [18]:
def compute_metrics_from_trades(trades: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    if trades.empty:
        return pd.DataFrame(), {
            "cumulative_return": np.nan,
            "annualized_sharpe": np.nan,
            "max_drawdown": np.nan,
            "avg_trade_fraction": np.nan,
            "num_trades": 0,
            "num_steps": 0,
            "num_markets": 0,
            "final_portfolio_value": np.nan,
        }

    trades = trades.sort_values(["market_id", "timestamp"]).reset_index(drop=True)

    trades["step_portfolio_return"] = (
        trades.groupby("market_id")["portfolio_value"]
        .pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    periods_per_year = (
        24 * 365
        if CONFIG["fidelity"] <= 60
        else max(int((60 / CONFIG["fidelity"]) * 24 * 365), 1)
    )

    per_market_metrics: List[Dict[str, float]] = []
    for market_id, g in trades.groupby("market_id", sort=False):
        g = g.sort_values("timestamp").reset_index(drop=True)

        initial_value = float(g["portfolio_value"].iloc[0])
        final_value = float(g["portfolio_value"].iloc[-1])
        episode_return = final_value / max(initial_value, 1e-8) - 1.0

        equity = g["portfolio_value"].to_numpy(dtype=float)
        running_peak = np.maximum.accumulate(equity)
        dd = equity / np.clip(running_peak, 1e-8, None) - 1.0
        episode_max_drawdown = float(dd.min())

        step_returns = g["step_portfolio_return"].to_numpy(dtype=float)
        step_returns = step_returns[np.isfinite(step_returns)]

        if len(step_returns) > 1 and np.std(step_returns, ddof=1) > 1e-12:
            episode_sharpe = float(
                np.sqrt(periods_per_year) * np.mean(step_returns) / np.std(step_returns, ddof=1)
            )
        else:
            episode_sharpe = np.nan

        executed_mask_g = g["trade_notional"] > 0
        avg_trade_fraction_g = float(
            (
                g.loc[executed_mask_g, "trade_notional"]
                / g.loc[executed_mask_g, "portfolio_value"].clip(lower=1e-8)
            ).mean()
        ) if executed_mask_g.any() else 0.0

        per_market_metrics.append({
            "market_id": market_id,
            "initial_value": initial_value,
            "final_value": final_value,
            "episode_return": episode_return,
            "episode_max_drawdown": episode_max_drawdown,
            "episode_sharpe": episode_sharpe,
            "num_steps": int(len(g)),
            "num_trades": int(executed_mask_g.sum()),
            "avg_trade_fraction": avg_trade_fraction_g,
            "total_fees_paid": float(g["fees_paid"].sum()),
        })

    per_market_df = pd.DataFrame(per_market_metrics)

    total_initial_value = float(per_market_df["initial_value"].sum())
    total_final_value = float(per_market_df["final_value"].sum())
    cumulative_return = total_final_value / max(total_initial_value, 1e-8) - 1.0

    pooled_step_returns = trades["step_portfolio_return"].to_numpy(dtype=float)
    pooled_step_returns = pooled_step_returns[np.isfinite(pooled_step_returns)]

    if len(pooled_step_returns) > 1 and np.std(pooled_step_returns, ddof=1) > 1e-12:
        annualized_sharpe = float(
            np.sqrt(periods_per_year) * np.mean(pooled_step_returns) / np.std(pooled_step_returns, ddof=1)
        )
    else:
        annualized_sharpe = np.nan

    executed_mask = trades["trade_notional"] > 0
    avg_trade_fraction = float(
        (
            trades.loc[executed_mask, "trade_notional"]
            / trades.loc[executed_mask, "portfolio_value"].clip(lower=1e-8)
        ).mean()
    ) if executed_mask.any() else 0.0

    metrics = {
        "cumulative_return": float(cumulative_return),
        "annualized_sharpe": annualized_sharpe,
        "max_drawdown": float(per_market_df["episode_max_drawdown"].min()),
        "avg_trade_fraction": avg_trade_fraction,
        "num_trades": int(executed_mask.sum()),
        "num_steps": int(len(trades)),
        "num_markets": int(trades["market_id"].nunique()),
        "final_portfolio_value": float(total_final_value),
    }

    return per_market_df, metrics


def evaluate_action_selector(
    action_selector,
    markets_dict: Dict[str, pd.DataFrame],
    rl_config: Dict[str, Any],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    env = make_eval_env(markets_dict)
    records: List[Dict[str, Any]] = []

    for _ in range(len(env.market_ids)):
        obs, info = env.reset()
        done = False
        step_idx = 0

        while not done:
            action = action_selector(obs, info, env, step_idx)
            next_obs, reward, terminated, truncated, step_info = env.step(action)
            done = terminated or truncated

            records.append({
                "market_id": step_info["market_id"],
                "timestamp": step_info["timestamp"],
                "yes_price": step_info["price"],
                "no_price": step_info["no_price"],
                "next_ret": step_info["next_ret"],
                "action_type": step_info["action_type"],
                "size_bin": step_info["size_bin"],
                "size_fraction": step_info["size_fraction"],
                "cash": step_info["cash"],
                "yes_qty": step_info["yes_qty"],
                "no_qty": step_info["no_qty"],
                "trade_notional": step_info["trade_notional"],
                "fees_paid": step_info["fees_paid"],
                "executed_qty": step_info["executed_qty"],
                "forced_liquidation_notional": step_info["forced_liquidation_notional"],
                "reward": reward,
                "risk_penalty": step_info["risk_penalty"],
                "trade_penalty": step_info["trade_penalty"],
                "portfolio_value": step_info["portfolio_value"],
            })

            obs = next_obs
            info = step_info
            step_idx += 1

    trades = pd.DataFrame(records)
    per_market_df, metrics = compute_metrics_from_trades(trades)
    return trades, per_market_df, metrics


def evaluate_agent(
    model: PPO,
    markets_dict: Dict[str, pd.DataFrame],
    rl_config: Dict[str, Any],
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    def model_selector(obs, info, env, step_idx):
        action, _ = model.predict(obs, deterministic=True)
        return action

    return evaluate_action_selector(model_selector, markets_dict, rl_config)


max_size_bin = RL_CONFIG["action_bins"] - 1

def hold_cash_selector(obs, info, env, step_idx):
    return np.array([0, 0], dtype=np.int64)

def buy_yes_and_hold_selector(obs, info, env, step_idx):
    if step_idx == 0:
        return np.array([1, max_size_bin], dtype=np.int64)
    return np.array([0, 0], dtype=np.int64)

def buy_no_and_hold_selector(obs, info, env, step_idx):
    if step_idx == 0:
        return np.array([2, max_size_bin], dtype=np.int64)
    return np.array([0, 0], dtype=np.int64)

def momentum_selector(obs, info, env, step_idx):
    if env.yes_qty <= 0.0 and env.no_qty <= 0.0:
        if step_idx == 0:
            return np.array([0, 0], dtype=np.int64)

        recent_ret = float(obs[RL_CONFIG["lookback"] + 3])
        if recent_ret > 0:
            return np.array([1, max_size_bin], dtype=np.int64)
        if recent_ret < 0:
            return np.array([2, max_size_bin], dtype=np.int64)
        return np.array([0, 0], dtype=np.int64)

    return np.array([0, 0], dtype=np.int64)


valid_trades, valid_per_market, valid_metrics = evaluate_agent(best_model, valid_markets, RL_CONFIG)
test_trades, test_per_market, test_metrics = evaluate_agent(best_model, test_markets, RL_CONFIG)

baseline_selectors = {
    "hold_cash": hold_cash_selector,
    "buy_yes_and_hold": buy_yes_and_hold_selector,
    "buy_no_and_hold": buy_no_and_hold_selector,
    "momentum": momentum_selector,
}

baseline_results: List[Dict[str, Any]] = []

for split_name, split_markets in [("valid", valid_markets), ("test", test_markets)]:
    for baseline_name, selector in baseline_selectors.items():
        _, _, baseline_metrics = evaluate_action_selector(selector, split_markets, RL_CONFIG)
        row = {"split": split_name, "strategy": baseline_name}
        row.update(baseline_metrics)
        baseline_results.append(row)

comparison_rows = [
    {"split": "valid", "strategy": "ppo_rl", **valid_metrics},
    {"split": "test", "strategy": "ppo_rl", **test_metrics},
]
comparison_rows.extend(baseline_results)
comparison_df = pd.DataFrame(comparison_rows)

valid_trades.to_csv(os.path.join(CONFIG["outdir"], "valid_rl_trades.csv"), index=False)
test_trades.to_csv(os.path.join(CONFIG["outdir"], "test_rl_trades.csv"), index=False)
valid_per_market.to_csv(os.path.join(CONFIG["outdir"], "valid_rl_per_market_metrics.csv"), index=False)
test_per_market.to_csv(os.path.join(CONFIG["outdir"], "test_rl_per_market_metrics.csv"), index=False)
comparison_df.to_csv(os.path.join(CONFIG["outdir"], "strategy_comparison.csv"), index=False)

with open(os.path.join(CONFIG["outdir"], "valid_rl_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(valid_metrics, f, indent=2)
with open(os.path.join(CONFIG["outdir"], "test_rl_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

print("Validation metrics")
display(valid_metrics)
print("Test metrics")
display(test_metrics)
print("Strategy comparison")
display(comparison_df.sort_values(["split", "strategy"]).reset_index(drop=True))

Validation metrics


{'cumulative_return': 0.18951695843684102,
 'annualized_sharpe': 0.8016882394922945,
 'max_drawdown': -0.6736154942335155,
 'avg_trade_fraction': 0.5828364604984102,
 'num_trades': 456,
 'num_steps': 36070,
 'num_markets': 461,
 'final_portfolio_value': 549825.9667301567}

Test metrics


{'cumulative_return': 0.1630767670885136,
 'annualized_sharpe': 1.2084001296148668,
 'max_drawdown': -0.74170828125,
 'avg_trade_fraction': 0.5988892152337931,
 'num_trades': 879,
 'num_steps': 36327,
 'num_markets': 471,
 'final_portfolio_value': 550226.7166567282}

Strategy comparison


,split,strategy,cumulative_return,annualized_sharpe,max_drawdown,avg_trade_fraction,num_trades,num_steps,num_markets,final_portfolio_value
0,test,buy_no_and_hold,0.008242,0.998076,-0.998919,1.007852,471,36327,471,472371.375737
1,test,buy_yes_and_hold,-0.039135,2.145606,-0.999408,1.009674,471,36327,471,456764.887731
2,test,hold_cash,0.000000,NaN,0.000000,0.000000,0,36327,471,471000.000000
3,test,momentum,-0.039656,1.510100,-0.999408,1.823372,445,36327,471,452322.126423
4,test,ppo_rl,0.163077,1.208400,-0.741708,0.598889,879,36327,471,550226.716657
5,valid,buy_no_and_hold,-0.005286,0.810277,-0.959070,1.003343,461,36070,461,461299.353583
6,valid,buy_yes_and_hold,0.069148,2.126553,-0.988649,1.006205,461,36070,461,501630.840455
7,valid,hold_cash,0.000000,NaN,0.000000,0.000000,0,36070,461,461000.000000
8,valid,momentum,-0.015168,1.261725,-0.988677,1.054969,435,36070,461,454007.757073
9,valid,ppo_rl,0.189517,0.801688,-0.673615,0.582836,456,36070,461,549825.966730
